### Import Libraries

In [ ]:
import os
import re
import sys
import json
import time
import csv
import yaml
import warnings
warnings.filterwarnings('ignore')
from langtrace_python_sdk import langtrace
LANGTRACE_API_KEY = os.environ["LANGTRACE_API_KEY"] = os.getenv("LANGTRACE_API_KEY")
langtrace.init(api_key=LANGTRACE_API_KEY)

Initializing Langtrace SDK..
⭐ Leave our github a star to stay on top of our updates - https://github.com/Scale3-Labs/langtrace
Exporting spans to Langtrace cloud..


# Agent: Market Analyst
## Task: Given an input domain or specific product/framework (e.g., Watsonx, CrewAI, Generative AI, Cloud, Automation), collect relevant industry trends. Extract trending skills from job postings, industry reports, and competitors. Identify emerging skills, tools & technologies, most popular libraries and frameoworks being used in that work, and high-demand roles. Compare industry demands with current company workforce skills.
Industry/Domain: Generative AI, IBM Client Engineering WatsonX



# Agent: Market Analyst
## Thought: Thought: I need to gather information about trending skills, high-demand roles, emerging technologies, and industry shifts in Generative AI, specifically within IBM Client Engineering WatsonX. I will use the Tavily Search Tool for this purpose.
## Using tool: Tavily Search Tool
## Tool Input: 
"{\"query\": \"trending skills in Generative AI IBM WatsonX high-demand roles emerging technologies industry shifts\"}"
## Tool Output: 
[{'url': 'h

In [2]:
from typing import List, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import (ScrapeWebsiteTool, WebsiteSearchTool, SerperDevTool, FileReadTool, FileWriterTool, CSVSearchTool, 
                          CodeInterpreterTool, NL2SQLTool, PDFSearchTool)
from crewai.tools import tool, BaseTool
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
from crewai.knowledge.source.csv_knowledge_source import CSVKnowledgeSource
from crewai.knowledge.source.crew_docling_source import CrewDoclingSource
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource

### Loading Tasks and Agents YAML Files

In [3]:
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

agents_config = configs['agents']
tasks_config = configs['tasks']

### Initialize LLMs

In [4]:
WATSONX_URL = os.environ["WATSONX_URL"] = os.getenv("WATSONX_URL")  
WATSONX_APIKEY = os.environ["WATSONX_APIKEY"] = os.getenv("WATSONX_APIKEY") 
WATSONX_PROJECT_ID = os.environ["WATSONX_PROJECT_ID"] = os.getenv("WATSONX_PROJECT_ID")
WATSONX_MODEL_ID = os.environ["WATSONX_MODEL_ID"] = "watsonx/ibm/granite-3-2-8b-instruct-preview-rc"    # ibm/granite-3-8b-instruct , ibm/granite-3-2-8b-instruct-preview-rc, mistralai/mistral-large
HF_TOKEN = os.environ["HUGGINGFACE_ACCESS_TOKEN"] = os.getenv("HF_TOKEN")

wx_llm = LLM(
    model = WATSONX_MODEL_ID,
    base_url = WATSONX_URL,
    project_id = WATSONX_PROJECT_ID,
	api_key = WATSONX_APIKEY,
    max_tokens = 10000,
    temperature = 0.1
)

### Define Tools

In [5]:
web_search_tool = WebsiteSearchTool()
serper_dev_tool = SerperDevTool()
file_read_tool = FileReadTool()
file_write_tool = FileWriterTool()

In [6]:
from pydantic import BaseModel, Field
from typing import List, Dict, Optional

class MarketTrendsOutput(BaseModel):
    trending_skills: List[str] = Field(..., description="List of in-demand skills in the given domain.")
    high_demand_roles: List[str] = Field(..., description="List of job roles experiencing high demand.")
    emerging_technologies: List[str] = Field(..., description="New and trending tools & technologies.")
    industry_shifts: str = Field(..., description="Summary of industry changes affecting workforce skills.")

class SkillGapReport(BaseModel):
    employee_id: str = Field(..., description="Unique identifier for the employee.")
    current_skills: List[str] = Field(..., description="Skills currently possessed by the employee.")
    missing_skills: List[str] = Field(..., description="Skills lacking compared to industry benchmarks.")
    role_deficiencies: Dict[str, List[str]] = Field(..., description="Skill gaps for specific job roles within the company.")
    priority_upskilling_areas: List[str] = Field(..., description="Areas where immediate training is required.")

class TrainingRecommendation(BaseModel):
    employee_id: str = Field(..., description="Unique identifier for the employee.")
    suggested_courses: List[str] = Field(..., description="List of recommended courses.")
    certifications: List[str] = Field(..., description="Relevant certifications for skill development.")
    mentorship_programs: Optional[List[str]] = Field(None, description="Mentorship initiatives for career growth.")
    estimated_timeframe: str = Field(..., description="Time required to acquire the recommended skills.")

class FinalReport(BaseModel):
    market_needs: MarketTrendsOutput = Field(..., description="report of market trends")
    skill_report: SkillGapReport = Field(..., description="skill gap report of employee")
    recommended_trainings: TrainingRecommendation = Field(..., description="recommended trainings and learnings")


### Define Agents

In [7]:
# json_source = JSONKnowledgeSource(
#     file_paths=["EMP12345.json"]
# )

In [ ]:
from pathlib import Path

courses_sample_file = "yl_courses.csv"
root_folder = Path("..").resolve()  
found_files = list(root_folder.rglob(courses_sample_file))
sample_courses = str(found_files[0])

csv_tool = CSVSearchTool(
    csv = sample_courses,
    config=dict(
        llm=dict(
            provider="huggingface",  # ('openai', 'azure_openai', 'anthropic', 'huggingface', 'cohere', 'together', 'gpt4all', 'ollama', 'jina', 'llama2', 'vertexai', 'google', 'aws_bedrock', 'mistralai', 'clarifai', 'vllm', 'groq', 'nvidia') 
            config=dict(
                model="ibm-granite/granite-3.1-8b-instruct",
            ),
        ),
        embedder=dict(
            provider="huggingface", 
            config=dict(
                model="ibm-granite/granite-embedding-30m-english"
            ),
        ),
    )
)

In [9]:
from langchain_community.tools.tavily_search import TavilySearchResults

class TavilyTool(BaseTool):
    name: str = "Tavily Search Tool"
    description: str = "Useful search engine built specifically for AI agents"
    search: TavilySearchResults = Field(default_factory=TavilySearchResults)
    
    def _run(self, query: str) -> str:
        try:
            return self.search.invoke(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"

In [10]:
market_trends_analyst = Agent(
  config = agents_config['market_trends_analyst'],
  verbose = True,
  llm = wx_llm,
  allow_delegation = False,
  tools = [web_search_tool, serper_dev_tool, TavilyTool()]
)

employee_skill_analyzer = Agent(
  config = agents_config['employee_skill_analyzer'],
  verbose = True,
  llm = wx_llm,
  allow_delegation = False
)

training_recommender = Agent(
  config = agents_config['training_recommender'],
  verbose = True,
  llm = wx_llm,
  allow_delegation = False,
  tools = [csv_tool]
)

agents_list = [market_trends_analyst, employee_skill_analyzer, training_recommender]

### Define Tasks

In [ ]:
analyze_market_trends_task = Task(
    config = tasks_config['analyze_market_trends_task'],
    agent = market_trends_analyst,
    output_pydantic = MarketTrendsOutput
)

skill_gap_detection_task = Task(
    config = tasks_config['skill_gap_detection_task'],
    agent = employee_skill_analyzer,
    context = [analyze_market_trends_task],
    output_pydantic = SkillGapReport
)

training_recommendation_task = Task(
    config = tasks_config['training_recommendation_task'],
    agent = training_recommender,
    output_pydantic = FinalReport,
    output_file = "report.json"
)

tasks_list = [analyze_market_trends_task, skill_gap_detection_task, training_recommendation_task]

### Run Crew

In [12]:
identify_talent_gaps = Crew(
    agents = agents_list, 
    tasks = tasks_list, 
    process = Process.sequential,
    verbose = True,
    # process=Process.hierarchical, 
)

In [13]:
employee_details = {
    "employee_id": "EMP12345",
    "name": "Alice Johnson",
    "band": "7B",
    "current_role": "Senior AI Engineer",
    "team": "Client Engineering WatsonX, IBM",
    "current_skills": ["Python", "SQL", "TensorFlow", "Data Analysis"],
    "manager_review": "Alice is proficient in data handling/engineering but needs exposure to advanced GenAI techniques.",
    "annual_reflection": "Good with predictive ML modeling and analysis but lacks hands-on experience with NLP and LLMs",
    "preferred_learning_style": ["In-House Company Courses", "Mentorship Programs"]
}

inputs = {
    "domain": "Generative AI, IBM Client Engineering WatsonX",
    "employee_details": employee_details
}

result = identify_talent_gaps.kickoff(inputs = inputs)

In [19]:
import pandas as pd
df_usage_metrics = pd.DataFrame([result.token_usage.dict()])
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")
df_usage_metrics

,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,7427,5964,0,1463,5
